# Parameter Sweep

The `sweep()` function runs `compute_stack` across a range of parameter values and collects results into a Polars DataFrame. This is useful for:

- **Energy optimization** — find the beam energy that maximizes yield for a target isotope
- **Current scaling** — verify linear response of activity vs. beam current
- **Thickness studies** — explore how target thickness affects production and purity
- **Irradiation time optimization** — find the optimal bombardment duration

## 1. Setup

In [ ]:
import numpy as np

from hyrr import (
    Beam, DataStore, Layer, TargetStack, sweep,
)
from hyrr.materials import resolve_element

db = DataStore("../nucl-parquet")

mo100 = resolve_element(db, "Mo", enrichment={100: 1.0})
beam = Beam(projectile="p", energy_MeV=16.0, current_mA=0.15)

base_stack = TargetStack(
    beam=beam,
    layers=[Layer(
        density_g_cm3=10.22,
        elements=[(mo100, 1.0)],
        energy_out_MeV=12.0,
    )],
    irradiation_time_s=86400.0,
    cooling_time_s=86400.0,
)

print("Setup complete.")

## 2. Sweep beam energy

Vary the incident proton energy from 12 to 24 MeV while keeping the exit energy at 12 MeV (i.e., increasing target thickness).

In [ ]:
energies = np.arange(13.0, 25.0, 0.5)

df_energy = sweep(db, base_stack, "beam.energy_MeV", energies)
df_energy.head(10)

In [ ]:
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(9, 5))

if "Tc-99m_activity_Bq" in df_energy.columns:
    ax1.plot(
        df_energy["param_value"].to_list(),
        [a * 1e-9 for a in df_energy["Tc-99m_activity_Bq"].to_list()],
        "o-", color="tab:blue", label="Tc-99m activity",
    )
    ax1.set_ylabel("Tc-99m Activity [GBq]", color="tab:blue")
    ax1.tick_params(axis="y", labelcolor="tab:blue")

ax1.set_xlabel("Beam Energy [MeV]")
ax1.set_title("p + Mo-100 → Tc-99m: Energy Sweep")

ax2 = ax1.twinx()
ax2.plot(
    df_energy["param_value"].to_list(),
    df_energy["total_heat_kW"].to_list(),
    "s--", color="tab:red", label="Heat",
)
ax2.set_ylabel("Total Heat [kW]", color="tab:red")
ax2.tick_params(axis="y", labelcolor="tab:red")

fig.tight_layout()
plt.show()

## 3. Sweep beam current

Activity should scale linearly with beam current.

In [ ]:
currents = np.linspace(0.05, 0.5, 10)

df_current = sweep(db, base_stack, "beam.current_mA", currents)

if "Tc-99m_activity_Bq" in df_current.columns:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(
        df_current["param_value"].to_list(),
        [a * 1e-9 for a in df_current["Tc-99m_activity_Bq"].to_list()],
        "o-",
    )
    ax.set_xlabel("Beam Current [mA]")
    ax.set_ylabel("Tc-99m Activity [GBq]")
    ax.set_title("Activity vs. Beam Current (should be linear)")
    ax.grid(True, alpha=0.3)
    plt.show()

## 4. Sweep irradiation time

Explore the build-up of activity as irradiation time increases. Tc-99m (t½ = 6 h) saturates relatively quickly.

In [ ]:
irr_times = np.linspace(3600, 48 * 3600, 20)  # 1 h to 48 h

df_irr = sweep(db, base_stack, "irradiation_time_s", irr_times)

if "Tc-99m_activity_Bq" in df_irr.columns:
    fig, ax = plt.subplots(figsize=(8, 5))
    hours = [t / 3600 for t in df_irr["param_value"].to_list()]
    activity_gbq = [a * 1e-9 for a in df_irr["Tc-99m_activity_Bq"].to_list()]
    ax.plot(hours, activity_gbq, "o-")
    ax.set_xlabel("Irradiation Time [hours]")
    ax.set_ylabel("Tc-99m Activity [GBq]")
    ax.set_title("Activity Saturation vs. Irradiation Time")
    ax.axhline(y=max(activity_gbq), color="gray", ls="--", alpha=0.5, label="~Saturation")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()

## 5. Sweep exit energy (effective thickness)

Lowering the exit energy means a thicker target — more production but also more heat and impurities.

In [ ]:
exit_energies = np.arange(6.0, 15.0, 0.5)

df_exit = sweep(db, base_stack, "layers[0].energy_out_MeV", exit_energies)
df_exit.head(10)

In [ ]:
if "Tc-99m_activity_Bq" in df_exit.columns:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(
        df_exit["param_value"].to_list(),
        [a * 1e-9 for a in df_exit["Tc-99m_activity_Bq"].to_list()],
        "o-",
    )
    ax1.set_xlabel("Exit Energy [MeV]")
    ax1.set_ylabel("Tc-99m Activity [GBq]")
    ax1.set_title("Activity vs. Exit Energy")
    ax1.invert_xaxis()  # lower exit = thicker target
    ax1.grid(True, alpha=0.3)

    ax2.plot(
        df_exit["param_value"].to_list(),
        df_exit["total_heat_kW"].to_list(),
        "s-", color="tab:red",
    )
    ax2.set_xlabel("Exit Energy [MeV]")
    ax2.set_ylabel("Total Heat [kW]")
    ax2.set_title("Heat Deposition vs. Exit Energy")
    ax2.invert_xaxis()
    ax2.grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()

## 6. Accessing all isotopes from a sweep

The sweep DataFrame contains activity columns for every isotope produced. You can inspect which columns are available and compare multiple isotopes.

In [ ]:
activity_cols = [c for c in df_energy.columns if c.endswith("_activity_Bq")]
print(f"Isotope columns ({len(activity_cols)}):")
for col in sorted(activity_cols):
    name = col.replace("_activity_Bq", "")
    max_val = df_energy[col].max()
    if max_val and max_val > 1e6:
        print(f"  {name:<12s}  max = {max_val:.4E} Bq")

## 7. Supported parameter paths

The `param` argument to `sweep()` accepts these dot-paths:

| Path | Description |
|------|-------------|
| `beam.energy_MeV` | Incident beam energy |
| `beam.current_mA` | Beam current |
| `irradiation_time_s` | Irradiation duration |
| `cooling_time_s` | Cooling duration |
| `layers[N].thickness_cm` | Layer N thickness |
| `layers[N].energy_out_MeV` | Layer N exit energy |
| `layers[N].areal_density_g_cm2` | Layer N areal density |